# PA2 baseline generation and full SFT

Run cells in order with the private reviewed training-data input, T4 x2 and Internet on. Cell 6 checks three independent development prompts. Inspect its output before running cell 7. Cell 7 saves all 300 pinned Alpaca baseline outputs, then cell 8 starts the full custom SFT loop. One epoch: 30,000 training / 1,000 validation examples; 938 planned optimizer updates. Watch cell 8 for update counts, loss, learning rate and validation loss. Do not rerun training into an existing output folder. Resume is unsupported. Best and final weights are saved separately. See bundled AI_USAGE.md for assistance.


In [ ]:
# 1. Verify both CUDA GPUs and record the starting environment.
import json, sys, subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import torch
output = Path('/kaggle/working/pa2_outputs')
output.mkdir(exist_ok=True)
info = {'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count(), 'packages': {}}
for name in ['transformers', 'datasets', 'accelerate', 'huggingface_hub', 'peft', 'trl', 'torchao']:
    try:
        info['packages'][name] = version(name)
    except PackageNotFoundError:
        info['packages'][name] = None
info['devices'] = []
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    info['devices'].append({'index': i, 'name': props.name, 'memory_gib': round(props.total_memory / 2**30, 2)})
    assert torch.ones(16, device=f'cuda:{i}').sum().item() == 16
print(json.dumps(info, indent=2))
assert info['cuda_available'] and info['gpu_count'] == 2, 'Two CUDA GPUs required'
assert all('T4' in d['name'] for d in info['devices']), 'Expected T4 devices'
# Keep the original starting environment if rerunning the setup cells.
initial = output / 'environment_before_setup.json'
if not initial.exists():
    initial.write_text(json.dumps(info, indent=2) + '\n')
print('Both T4 GPUs execute tensors.')


In [ ]:
# 2. Pin PA2 dependencies, preserve CUDA PyTorch and verify fresh Qwen imports.
torch_before = version('torch')
constraint = output / 'preserve_torch.txt'
constraint.write_text('torch==' + torch_before + '\n')
packages = ['transformers==4.57.6', 'datasets==4.8.5', 'accelerate==1.15.0', 'huggingface_hub==0.36.2',
            'tqdm', 'pyyaml', 'sentencepiece', 'safetensors', 'pytest>=8', 'matplotlib']
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                         '-c', str(constraint), *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(output / 'package_setup.log').write_text(result.stdout)
print(result.stdout[-3500:])
result.check_returncode()
assert version('torch') == torch_before, 'CUDA PyTorch must not change'
probe_command = [sys.executable, '-c', 'from transformers import AutoModelForCausalLM, AutoTokenizer; from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM; print("Qwen model imports pass")']
probe = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if probe.returncode and 'torchao' in probe.stdout:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    probe = subprocess.run(probe_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(probe.stdout[-3500:])
probe.check_returncode()
versions = {name: version(name) for name in ['torch', 'transformers', 'datasets', 'accelerate', 'huggingface_hub']}
(output / 'environment_after_setup.json').write_text(json.dumps(versions, indent=2) + '\n')
print(json.dumps(versions, indent=2))
print('PA2 dependencies ready. CUDA PyTorch preserved.')


In [ ]:
# 3. Restore the exact reviewed source snapshot; verify archive and file hashes.
import base64, hashlib, io, zipfile
BUNDLE_SHA256 = '4d55f6187e4708a60789394bb7c237c26e87bf664038a78faff6c2c987456ff8'
BUNDLE_B64 = '...'
bundle = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(bundle).hexdigest() == BUNDLE_SHA256, 'Source archive hash mismatch'
root = Path('/kaggle/working') / ('pa2-reviewed-' + BUNDLE_SHA256[:12])
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for item in archive.infolist():
        name = Path(item.filename)
        assert not name.is_absolute() and '..' not in name.parts, 'Invalid archive path'
        target = root / name
        data = archive.read(item)
        if target.exists():
            assert target.is_file() and target.read_bytes() == data, 'Existing snapshot differs: ' + str(name)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
source_manifest = json.loads((root / 'source_manifest.json').read_text())
for name, expected in source_manifest['files'].items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, 'File hash mismatch: ' + name
(output / 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2) + '\n')
(output / 'pa2-reviewed-source.zip').write_bytes(bundle)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(root), '--no-deps'], check=True)
print('Reviewed source installed:', root)
print('Source archive SHA256:', BUNDLE_SHA256)
print('Verified source files:', len(source_manifest['files']))


In [ ]:
# 4. Run the independent suite on Kaggle (MPS-specific check skips on CUDA).
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests', '--tb=short'], cwd=root,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(output / 'tests.log').write_text(result.stdout)
print(result.stdout)
result.check_returncode()
print('Kaggle suite passed.')


In [ ]:
# 5. Verify every prepared input file before any model downloads.
expected_zip = '1360f4f66f5667d497275b13ebfd3b77092826d0c544bd9f1eb9a3452d607023'
expected_manifest = '03887a5d65a366e8ed9df3789e5aa16a56ca1a5a914da7a9277ffcc1f969363c'
banks = list(Path('/kaggle/input').rglob('bundle_manifest.json'))
if not banks:
    archives = list(Path('/kaggle/input').rglob('pa2_prepared_training_data.zip'))
    assert len(archives) == 1, 'Attach the reviewed private training data input'
    assert hashlib.sha256(archives[0].read_bytes()).hexdigest() == expected_zip
    data_root = Path('/kaggle/working/pa2_training_data')
    data_root.mkdir(exist_ok=False)
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            name = Path(item.filename)
            assert not name.is_absolute() and '..' not in name.parts
            archive.extract(item, data_root)
    banks = [data_root / 'bundle_manifest.json']
assert len(banks) == 1, 'Ambiguous prepared input'
data_root = banks[0].parent
assert hashlib.sha256(banks[0].read_bytes()).hexdigest() == expected_manifest
data_manifest = json.loads(banks[0].read_text())
for name, entry in data_manifest['files'].items():
    path = data_root / name
    assert path.stat().st_size == entry['bytes']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == entry['sha256'], name
run_root = Path('/kaggle/working/pa2_runs')
run_root.mkdir(exist_ok=True)
print('Verified all prepared inputs:', data_root, flush=True)
def run_logged(command, name):
    with (run_root / name).open('x') as log:
        process = subprocess.Popen(command, cwd=root, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise RuntimeError('Run failed; preserved log: ' + name)


In [ ]:
# 6. Sanity check: inspect full prompts, new tokens, EOS and length before baseline.
run_logged([sys.executable, '-u', str(root / 'scripts/generate_baseline.py'),
            '--bank', str(data_root / 'alpaca_eval/eval.jsonl'),
            '--output-dir', str(run_root / 'base_sanity_controls.json'), '--sanity-only'], 'base_sanity_controls.log')


In [ ]:
# 7. Generate and persist all 300 base-model Alpaca responses.
run_logged([sys.executable, '-u', str(root / 'scripts/generate_baseline.py'),
            '--bank', str(data_root / 'alpaca_eval/eval.jsonl'),
            '--output-dir', str(run_root / 'base_alpaca')], 'base_generation.log')
assert json.loads((run_root / 'base_alpaca/run.json').read_text())['completed'] == 300
print('BASELINE COMPLETE: 300/300. Full SFT is next.', flush=True)


In [ ]:
# 8. FULL SFT: one epoch, 30k train / 1k validation, 938 planned updates.
# Visible logs show updates every 10 steps and validation every 50.
assert json.loads((run_root / 'base_alpaca/run.json').read_text())['status'] == 'completed'
run_logged([sys.executable, '-u', str(root / 'scripts/train_qwen.py'),
            '--track', 'sft', '--data-dir', str(data_root / 'sft'),
            '--output-dir', str(run_root / 'sft_full'), '--full-run'], 'sft_full.log')
print('FULL SFT COMPLETE. Review validation curves and development outputs before DPO.', flush=True)
